# G1 URDF — 43 Revolute Joint Coordinate Systems

Renders the G1 robot kinematic skeleton from `g1_29dof_with_hand_rev_1_0_pkg.urdf`
at the zero configuration (**q = 0**) and overlays all **43 revolute joint frames**.

| Group | Joints | DOF |
|---|---|---|
| Hip / thigh (L + R) | pitch, roll, yaw × 2 | 6 |
| Knee (L + R) | 1 × 2 | 2 |
| Ankle (L + R) | pitch, roll × 2 | 4 |
| Waist | yaw, roll, pitch | 3 |
| Shoulder (L + R) | pitch, roll, yaw × 2 | 6 |
| Elbow (L + R) | 1 × 2 | 2 |
| Wrist (L + R) | roll, pitch, yaw × 2 | 6 |
| Dex3 thumb (L + R) | 0 (rot), 1, 2 × 2 | 6 |
| Dex3 middle (L + R) | 0, 1 × 2 | 4 |
| Dex3 index (L + R) | 0, 1 × 2 | 4 |
| **Total** | | **43** |

**Frame convention:** `X = red` · `Y = green` · `Z = blue` · thick arrow = joint rotation axis.

> **Requirements:** `numpy`, `matplotlib`. Install with `pip install numpy matplotlib`.


In [ ]:
from __future__ import annotations

import sys
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

try:
    import numpy as np
except ImportError:
    sys.exit("numpy not found — run:  pip install numpy")

try:
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    from matplotlib.lines import Line2D
except ImportError:
    sys.exit("matplotlib not found — run:  pip install matplotlib")

# interactive 3-D widget if ipympl is available, else inline PNG
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
    print("interactive 3-D (ipympl)")
except Exception:
    try:
        get_ipython().run_line_magic("matplotlib", "inline")
        print("static inline PNG")
    except Exception:
        pass

print(f"numpy      {np.__version__}")
print(f"matplotlib {plt.matplotlib.__version__}")


In [ ]:
# ── URDF path detection ──────────────────────────────────────────────────────
NB_DIR  = Path.cwd()
G1_ROOT = NB_DIR.parent.parent   # sim/notebooks/ → sim/ → g1/

URDF_CANDIDATES = [
    G1_ROOT / "install/g1_description/share/g1_description/urdf/g1_29dof_with_hand_rev_1_0_pkg.urdf",
    G1_ROOT / "sim/G1_rviz_simulation-main/G1_rviz_simulation-main/install/g1_description/share/g1_description/urdf/g1_29dof_with_hand_rev_1_0_pkg.urdf",
    # Termux path
    Path("/data/data/com.termux/files/home/ef_ws/g1/install/g1_description/share/g1_description/urdf/g1_29dof_with_hand_rev_1_0_pkg.urdf"),
    # Jetson / ROS2 workspace path
    Path("/home/unitree/EF/ef_ws/g1/install/g1_description/share/g1_description/urdf/g1_29dof_with_hand_rev_1_0_pkg.urdf"),
]

URDF_PATH = next((p for p in URDF_CANDIDATES if p.exists()), None)
if URDF_PATH is None:
    searched = "\n".join(f"  {p}" for p in URDF_CANDIDATES)
    raise FileNotFoundError(
        f"Cannot find G1 URDF.\nAdd your path to URDF_CANDIDATES above.\nSearched:\n{searched}"
    )
print(f"URDF: {URDF_PATH}")


In [ ]:
# ── Parse URDF ───────────────────────────────────────────────────────────────

def _fv(s: str) -> list:
    return [float(x) for x in s.strip().split()]


def parse_urdf(path: Path):
    root = ET.parse(path).getroot()
    joints: dict = {}
    children: dict = {}

    for jel in root.findall("joint"):
        name       = jel.get("name")
        jtype      = jel.get("type")
        par_link   = jel.find("parent").get("link")
        chi_link   = jel.find("child").get("link")
        orig       = jel.find("origin")
        xyz  = _fv(orig.get("xyz", "0 0 0")) if orig is not None else [0., 0., 0.]
        rpy  = _fv(orig.get("rpy", "0 0 0")) if orig is not None else [0., 0., 0.]
        axel = jel.find("axis")
        axis = _fv(axel.get("xyz", "0 0 1")) if axel is not None else [0., 0., 1.]
        joints[name] = {
            "type":   jtype,
            "parent": par_link,
            "child":  chi_link,
            "xyz":    xyz,
            "rpy":    rpy,
            "axis":   axis,
        }
        children.setdefault(par_link, []).append(name)

    return joints, children


joints, children = parse_urdf(URDF_PATH)
revolute = {n: d for n, d in joints.items() if d["type"] == "revolute"}

print(f"Total joints : {len(joints)}")
print(f"  revolute   : {len(revolute)}")
print(f"  fixed      : {sum(1 for d in joints.values() if d['type'] == 'fixed')}")


In [ ]:
# ── Forward Kinematics at q = 0 ──────────────────────────────────────────────

def rpy_to_R(rpy) -> np.ndarray:
    r, p, y = rpy
    Rx = np.array([[1, 0,          0         ],
                   [0, np.cos(r), -np.sin(r) ],
                   [0, np.sin(r),  np.cos(r) ]])
    Ry = np.array([[ np.cos(p), 0, np.sin(p)],
                   [ 0,         1, 0        ],
                   [-np.sin(p), 0, np.cos(p)]])
    Rz = np.array([[np.cos(y), -np.sin(y), 0],
                   [np.sin(y),  np.cos(y), 0],
                   [0,          0,         1]])
    return Rz @ Ry @ Rx


def make_T(xyz, rpy) -> np.ndarray:
    T = np.eye(4)
    T[:3, :3] = rpy_to_R(rpy)
    T[:3,  3] = np.asarray(xyz)
    return T


def fk(joints: dict, children: dict, root: str = "pelvis") -> dict:
    # BFS/DFS forward kinematics at zero joint angles
    T_world: dict = {root: np.eye(4)}

    def _visit(link: str) -> None:
        for jname in children.get(link, []):
            j = joints[jname]
            T_world[j["child"]] = T_world[link] @ make_T(j["xyz"], j["rpy"])
            _visit(j["child"])

    _visit(root)
    return T_world


T_world = fk(joints, children)
print(f"Transforms computed: {len(T_world)} links")
print(f"  pelvis (root)      : {T_world['pelvis'][:3,3].round(4)}")
print(f"  torso_link         : {T_world.get('torso_link', np.eye(4))[:3,3].round(4)}")
print(f"  left_ankle_roll    : {T_world.get('left_ankle_roll_link',  np.eye(4))[:3,3].round(4)}")
print(f"  right_ankle_roll   : {T_world.get('right_ankle_roll_link', np.eye(4))[:3,3].round(4)}")


In [ ]:
# ── Joint groups and colors ───────────────────────────────────────────────────
# Keys are substrings matched against joint names (first match wins).
GROUP_COLORS: dict = {
    "left_ankle":     "#00BFFF",   # sky blue
    "left_knee":      "#0080FF",   # blue
    "left_hip":       "#0000CC",   # dark blue
    "right_ankle":    "#FFD700",   # gold
    "right_knee":     "#FFA500",   # orange
    "right_hip":      "#FF4500",   # red-orange
    "waist":          "#00CC44",   # green
    "left_shoulder":  "#CC0000",   # dark red
    "left_elbow":     "#FF4444",   # red
    "left_wrist":     "#FF9999",   # light red / pink
    "right_shoulder": "#8800CC",   # purple
    "right_elbow":    "#CC44CC",   # magenta
    "right_wrist":    "#FF88FF",   # light magenta
    "left_hand":      "#3333FF",   # indigo
    "right_hand":     "#AA44FF",   # violet
}


def joint_style(name: str):
    for group, color in GROUP_COLORS.items():
        if group in name:
            return group, color
    return "other", "#888888"


for jname, jdata in revolute.items():
    g, c = joint_style(jname)
    jdata["group"] = g
    jdata["color"] = c

counts = Counter(d["group"] for d in revolute.values())
print(f"{'Group':<22} {'DOF':>4}")
print("-" * 28)
for g in GROUP_COLORS:
    if g in counts:
        print(f"  {g:<20} {counts[g]:>4}")
print("-" * 28)
print(f"  {'TOTAL':<20} {sum(counts.values()):>4}")


In [ ]:
# ── 3-D Visualization ────────────────────────────────────────────────────────
FRAME_SCALE = 0.045   # axis arrow length in metres
BONE_LW     = 2.2     # skeleton line width
BONE_ALPHA  = 0.28    # skeleton opacity

fig = plt.figure(figsize=(20, 13))
ax  = fig.add_subplot(111, projection="3d")
ax.set_facecolor("#0e0e12")
fig.patch.set_facecolor("#0e0e12")

# ── (a) Skeleton segments ─────────────────────────────────────────────────────
for jname, jdata in joints.items():
    if jdata["type"] == "fixed":
        continue
    p_link, c_link = jdata["parent"], jdata["child"]
    if p_link not in T_world or c_link not in T_world:
        continue
    p0 = T_world[p_link][:3, 3]
    p1 = T_world[c_link][:3, 3]
    _, color = joint_style(jname)
    ax.plot([p0[0], p1[0]], [p0[1], p1[1]], [p0[2], p1[2]],
            color=color, lw=BONE_LW, alpha=BONE_ALPHA)

# ── (b) Coordinate frames at each revolute joint ──────────────────────────────
_axis_colors = ("#CC0000", "#00AA00", "#0000CC")   # X, Y, Z

for jname, jdata in revolute.items():
    c_link = jdata["child"]
    if c_link not in T_world:
        continue
    T = T_world[c_link]
    o = T[:3, 3]    # joint origin in world frame
    R = T[:3, :3]   # joint orientation

    # X, Y, Z arrows (thin, canonical colors)
    for col_idx, rgb in enumerate(_axis_colors):
        d = R[:, col_idx] * FRAME_SCALE
        ax.quiver(o[0], o[1], o[2], d[0], d[1], d[2],
                  color=rgb, linewidth=1.1, arrow_length_ratio=0.32, alpha=0.92)

    # Joint rotation axis (thick, body-part color)
    ja_w = R @ np.array(jdata["axis"])
    ax.quiver(o[0], o[1], o[2],
              ja_w[0] * FRAME_SCALE * 1.6,
              ja_w[1] * FRAME_SCALE * 1.6,
              ja_w[2] * FRAME_SCALE * 1.6,
              color=jdata["color"], linewidth=2.6,
              arrow_length_ratio=0.38, alpha=0.97)

# ── (c) Joint labels ──────────────────────────────────────────────────────────
for jname, jdata in revolute.items():
    c_link = jdata["child"]
    if c_link not in T_world:
        continue
    pos = T_world[c_link][:3, 3]
    short = (jname
             .replace("_joint", "")
             .replace("left_",  "L.")
             .replace("right_", "R.")
             .replace("_pitch", ".P")
             .replace("_roll",  ".Ro")
             .replace("_yaw",   ".Y"))
    ax.text(pos[0], pos[1], pos[2], short,
            fontsize=5, color=jdata["color"],
            ha="center", va="bottom", alpha=0.92)

# ── (d) Legends ───────────────────────────────────────────────────────────────
body_handles = [
    Line2D([0], [0], color=c, lw=3, label=g.replace("_", " "))
    for g, c in GROUP_COLORS.items()
]
axis_handles = [
    Line2D([0], [0], color="#CC0000", lw=2,   label="X axis"),
    Line2D([0], [0], color="#00AA00", lw=2,   label="Y axis"),
    Line2D([0], [0], color="#0000CC", lw=2,   label="Z axis"),
    Line2D([0], [0], color="#AAAAAA", lw=2.5, linestyle="--",
           label="joint rotation axis"),
]
leg1 = ax.legend(handles=body_handles, loc="upper left",
                 fontsize=6.5, ncol=2, framealpha=0.7,
                 title="Body part", facecolor="#1a1a22",
                 labelcolor="white", title_fontsize=7)
leg1.get_title().set_color("white")
ax.add_artist(leg1)
leg2 = ax.legend(handles=axis_handles, loc="upper right",
                 fontsize=7, framealpha=0.7,
                 title="Frame axes", facecolor="#1a1a22",
                 labelcolor="white", title_fontsize=7)
leg2.get_title().set_color("white")

# ── (e) Styling and view ──────────────────────────────────────────────────────
for spine in ["left", "right", "top", "bottom"]:
    ax.spines[spine].set_visible(False) if hasattr(ax, "spines") else None

ax.set_xlabel("X (m)", color="white", fontsize=9)
ax.set_ylabel("Y (m)", color="white", fontsize=9)
ax.set_zlabel("Z (m)", color="white", fontsize=9)
ax.tick_params(colors="white", labelsize=7)
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
ax.xaxis.pane.set_edgecolor("#333340")
ax.yaxis.pane.set_edgecolor("#333340")
ax.zaxis.pane.set_edgecolor("#333340")
ax.grid(True, color="#333340", linewidth=0.5)

ax.set_title(
    "G1 URDF — 43 Revolute Joint Coordinate Systems  (q = 0)",
    color="white", fontsize=13, fontweight="bold", pad=12
)

# Equal aspect ratio
pts  = np.stack([T_world[d["child"]][:3, 3]
                 for d in revolute.values() if d["child"] in T_world])
mn, mx = pts.min(0), pts.max(0)
mid    = (mn + mx) / 2
half   = (mx - mn).max() / 2 + 0.14
ax.set_xlim(mid[0] - half, mid[0] + half)
ax.set_ylim(mid[1] - half, mid[1] + half)
ax.set_zlim(mid[2] - half, mid[2] + half)

ax.view_init(elev=14, azim=-68)
plt.tight_layout()

SAVE_PATH = Path("g1_joint_frames.png")
plt.savefig(SAVE_PATH, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print(f"Saved → {SAVE_PATH.resolve()}")


In [ ]:
# ── Joint summary table ────────────────────────────────────────────────────────
try:
    import pandas as pd

    rows = []
    for i, (jname, jdata) in enumerate(revolute.items(), 1):
        c_link = jdata["child"]
        T      = T_world.get(c_link, np.eye(4))
        pos    = T[:3, 3]
        ja_l   = jdata["axis"]               # joint axis in local frame
        ja_w   = (T[:3, :3] @ np.array(ja_l)).round(3)
        rows.append({
            "#":             i,
            "joint":         jname,
            "group":         jdata["group"],
            "child link":    c_link,
            "x (m)":         round(float(pos[0]), 4),
            "y (m)":         round(float(pos[1]), 4),
            "z (m)":         round(float(pos[2]), 4),
            "axis local":    " ".join(f"{v:.0f}" for v in ja_l),
            "axis world":    " ".join(f"{v:+.2f}" for v in ja_w),
        })

    df = pd.DataFrame(rows).set_index("#")
    display(df)

except ImportError:
    print("pandas not installed (pip install pandas).  Plain output:")
    print(f"{'#':>3}  {'joint':<45}  {'group':<20}  {'pos (m)'}")
    print("-" * 100)
    for i, (jname, jdata) in enumerate(revolute.items(), 1):
        c_link = jdata["child"]
        pos    = T_world.get(c_link, np.eye(4))[:3, 3].round(4)
        print(f"{i:3d}  {jname:<45}  {jdata['group']:<20}  {pos.tolist()}")
